# 📄 On the Superlinear Relationship between SGD Noise Covariance and Loss Landscape Curvature

| 项目 | 内容 |
| --- | --- |
| **作者** | Yikuan Zhang*, Ning Yang*, Yuhai Tu（北大物理学院 / Flatiron Institute） |
| **年份 / 出处** | 2026, arXiv 预印本 |
| **链接** | [arXiv:2602.05600](https://arxiv.org/abs/2602.05600) · [代码](https://anonymous.4open.science/r/AWCH-1A6A) |
| **阅读日期** | 2026-07-18 |
| **标签** | `SGD噪声` `损失景观` `Hessian` `隐式正则化` |
| **推荐指数** |  |

## 🎯 一句话总结（TL;DR）

> 经典观点认为 SGD 噪声协方差 $C \propto H$（基于 Fisher $\approx$ Hessian），本文指出该论证的前提在深度网络中普遍不成立；改用 **Activity–Weight Duality (AWD)** 推出与损失形式无关的关系 $C \propto \mathbb{E}_p[h_p^2]$（逐样本 Hessian 的**二阶矩**，而 $H = \mathbb{E}_p[h_p]$ 是一阶矩），从而 $C$ 与 $H$ 只是**近似可交换**而非成比例，对角元满足幂律 $C_{ii} \propto H_{ii}^{\gamma}$，且理论上界定 $1 \le \gamma \le 2$：MSE 损失 $\gamma \approx 1$，交叉熵损失显著超线性（$\gamma$ 可达 $1.4{\sim}1.5$）。

## ❓ 三个问题

### WHY — 要解决什么问题？

- SGD 噪声的**各向异性**（$C$ 与 $H$ 对齐）被认为是逃离尖锐极小、偏向平坦极小的关键机制，但其定量刻画依赖 "Fisher $=$ Hessian $\Rightarrow C \propto H$" 这条老论证。
- 该论证需要：负对数似然损失 + 可实现性（realizability，模型分布 $=$ 真实数据分布）+ 似然的 $C^2$ 光滑性。实际中：真实标签分布是 Dirac delta $p_{\text{data}}(y|x) = \delta(y - y^*(x))$，抵消机制失效；ReLU/max-pooling 破坏光滑性；MSE 根本不适用。
- 还有一个干净的**量纲反驳**：$[H] = [L]/[w]^2$，$[C] = [L]^2/[w]^2$，若 $C \propto H$ 则比例常数必须带损失量纲 $[L]$——关系会依赖损失的任意标度，不可能是普适的。

### HOW — 用了什么方法？

- **AWD（Feng et al., 2023, Nat. Mach. Intell.）**：把 mini-batch 间的数据（activity）扰动 $\Delta a$ 精确映射为等效的权重扰动 $\Delta W^*$（保持前激活不变、Frobenius 范数最小），与损失函数形式无关。
- 用 AWD 把梯度差 $g_{\mu\nu}$ 化为 Hessian 驱动项 $\frac{1}{B}\sum_p h_p \Delta w_p$，进而 $C \approx \frac{\sigma_w^2}{2B}\, \mathbb{E}_p[h_p^2]$。
- 在 $H$ 的特征基下做谱分解，比较一阶矩（$H_{ii}$）与二阶矩（$C_{ii}$），并用逐样本 Hessian 的 PSD 性 + Jensen/Cauchy–Schwarz 证明 $1 \le \gamma \le 2$。
- 实验：MNIST / CIFAR-10 × MLP / CNN × MSE / CE，测经验 $\gamma_{\text{emp}}$ 与 AWD 预测 $\gamma_{\text{AWD}}$；再用"压制实验"定位 CE 与 MSE 差异的几何来源。

### WHAT — 得到了什么结论？

- $C$ 与 $H$ **近似共特征系**（可交换 $[C,H]\approx 0$）：$H$ 基下的相关矩阵 $R$ 高度对角化（$\mu_{\text{real}} \approx 0.066$，远低于随机基线 $\mu_{\text{rand}} \approx 0.2$）。
- $C_{ii} \propto H_{ii}^{\gamma}$：MSE 的 $\gamma \approx 1$，CE 的 $\gamma \in [1.08, 1.51]$（CIFAR-10 + MLP + 10 类时最高），$\gamma_{\text{AWD}}$ 与 $\gamma_{\text{emp}}$ 高度吻合——直接反驳 Fisher 近似预测的严格线性。
- **超线性的几何起源**：逐样本 Hessian 由第一特征值主导，$C_{ii} \propto \mathbb{E}_p[XY]$，其中 $X = (\kappa_1^{(p)})^2$（局部曲率强度）、$Y = (u_1^{(p)} \cdot v_i)^2$（局部-全局对齐度）。CE 中 $X$ 与 $Y$ 正相关 $\Rightarrow \gamma > 1$；MSE 中二者近似独立 $\Rightarrow \gamma \to 1$（压制实验：把 $\kappa_1$ 换成均值后 CE 的谱明显畸变、MSE 几乎不变）。
- 幂律在训练早期（远未收敛时）就已出现，说明噪声-曲率对齐是 SGD 的内禀性质，而非渐近极限现象。

## 🧠 核心思想与主要结论

**设定。** 训练集 $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$，SGD 更新 $w_{t+1} = w_t - \eta \nabla L_{\mu_t}(w_t)$。CLT 下 $\nabla L_{\mu}(w) \sim \mathcal{N}(g(w), C(w))$，收敛邻域内 $\|g\| \approx 0$，噪声协方差由未中心化二阶矩主导：

$$
C(w) \approx \frac{1}{BN} \sum_{i=1}^{N} \nabla \ell_i(w)\, \nabla \ell_i(w)^{\top}.
$$

**核心结果（Theorem 3.4 + Theorem 5.1）。** 记逐样本 Hessian $h_p$，其谱分解 $h_p u_m^{(p)} = \kappa_m^{(p)} u_m^{(p)}$；全局 Hessian $H = \mathbb{E}_p[h_p]$，特征基 $\{v_i\}$。在 AWD 梯度近似 + 扰动各向同性假设下：

$$
C_{ij} \approx \frac{\sigma_w^2}{2B}\, \mathbb{E}_p\!\left[ \sum_m \big(\kappa_m^{(p)}\big)^2 \big(u_m^{(p)} \!\cdot v_i\big) \big(u_m^{(p)} \!\cdot v_j\big) \right],
\qquad
H_{ii} = \mathbb{E}_p\!\left[ \sum_m \kappa_m^{(p)} \big(u_m^{(p)} \!\cdot v_i\big)^2 \right].
$$

- $H$ 是局部曲率的**一阶矩**，$C$ 是**二阶矩**——重尾谱下二阶矩衰减更快，自然给出超线性。
- 高维下 $u_m^{(p)}$ 在正交方向 $v_i, v_j$ 上的投影近似独立零均值 $\Rightarrow$ 非对角元 $C_{ij}$ 渐近消失 $\Rightarrow [C, H] \approx 0$。
- 量纲自洽：$[C] = [L]^2/[w]^4 \cdot [w]^2 = [L]^2/[w]^2$ ✓（对比 Fisher 论证的量纲矛盾）。
- 局部极小邻域（$h_p$ PSD、特征值 $\le \kappa_{\max}$）内有双侧界：

$$
\frac{\sigma_w^2}{2B} H_{ii}^2 \;\le\; C_{ii} \;\le\; \frac{\sigma_w^2 \kappa_{\max}}{2B} H_{ii}
\quad\Longrightarrow\quad 1 \le \gamma \le 2.
$$

**两个极限情形（Prop 5.3 / 5.5）：**
- $\gamma \to 2$：局部特征基与全局完全对齐（$(u_m^{(p)} \cdot v_i)^2 = \delta_{mi}$）且曲率涨落 $\mathrm{Var}(\kappa_i^{(p)}) \propto (\mathbb{E}[\kappa_i^{(p)}])^2$，则 $C_{ii} \propto H_{ii}^2$。现实高维景观中不可实现，故 $\gamma = 2$ 只是理论上界。
- $\gamma \to 1$：谱退化（前 $n$ 个特征值同量级 $\bar\kappa^{(p)}$，与方向无关），则 $C_{ii} \propto \frac{\mu_{\kappa^2}}{\mu_\kappa} H_{ii}$，比例常数就是曲率的二阶矩/一阶矩。

## ✏️ 公式推导

### 推导 1：Fisher 近似为何失效

对 $\ell = -\log p_w(y|x)$，逐项求二阶导：

$$
\begin{aligned}
\nabla^2 \big(-\log p_w\big)
&= -\nabla \left( \frac{\nabla p_w}{p_w} \right)
= \frac{\nabla p_w \nabla p_w^{\top}}{p_w^2} - \frac{\nabla^2 p_w}{p_w} \\
&= \underbrace{\nabla \log p_w \, \nabla \log p_w^{\top}}_{\text{经验 Fisher } F \text{ 的被加项}} \; - \; \frac{\nabla^2 p_w}{p_w}.
\end{aligned}
$$

对样本取平均得 $H = F - \frac{1}{N} \sum_i \frac{\nabla^2 p_w(y_i|x_i)}{p_w(y_i|x_i)}$。第二项要在期望下消失，需 $y \sim p_{w^*}$（可实现性）：

$$
\mathbb{E}_{x, y \sim p_{\text{data}}}\!\left[ \frac{\nabla^2 p_{w^*}(y|x)}{p_{w^*}(y|x)} \right]
= \int p(x) \, \nabla^2 \underbrace{\int p_{w^*}(y|x)\, dy}_{=1} \, dx = 0,
$$

其中关键一步是 $p_{\text{data}} = p_{w^*}$ 时被积函数中的 $p_{w^*}$ 恰好约掉。但真实数据 $p_{\text{data}}(y|x) = \delta(y - y^*(x))$ 时，期望塌缩为单点取值 $\frac{\nabla^2 p_w(y^*|x)}{p_w(y^*|x)}$，积分-求导交换的抵消机制不复存在 $\Rightarrow H \ne F \Rightarrow C \propto H$ 失去依据。

### 推导 2：AWD 的秩-1 闭式解（Lemma 3.2）

约束 $\Delta W a = W \Delta a =: b$ 是对 $\Delta W$ 每一行的线性约束：$\Delta w_j^{\top} a = b_j$。最小范数解逐行独立：

$$
\min_{\Delta w_j} \|\Delta w_j\|^2 \;\; \text{s.t.} \;\; \Delta w_j^{\top} a = b_j
\quad\Longrightarrow\quad
\Delta w_j = \frac{b_j\, a}{\|a\|^2}
$$

（拉格朗日条件 $\Delta w_j \parallel a$，系数由约束定）。合并各行：

$$
\Delta W^* = \frac{b\, a^{\top}}{\|a\|^2} = \frac{(W \Delta a)\, a^{\top}}{\|a\|^2},
$$

即等效权重噪声 $=$ 传播误差 $W\Delta a$ 与输入激活 $a$ 的秩-1 外积。这一步把"数据抖动"翻译成"权重抖动"，且逐样本损失不变：$\ell(x_{p'}; W) = \ell(x_p; W + \Delta W)$。

### 推导 3：梯度差的 Hessian 主导近似（Lemma 3.3）

把 $\mathcal{B}_\mu$ 中的配对样本 $p'$ 看作 $p \in \mathcal{B}_\nu$ 的扰动版本，围绕 $w + \Delta w_p^{\mu\nu}$ 一阶 Taylor 展开：

$$
\begin{aligned}
\nabla \ell_p(w) - \nabla \ell_{p'}(w)
&\approx \nabla \ell_p(w) - \nabla \ell_p(w + \Delta w_p^{\mu\nu}) \\
&\approx -\underbrace{h_p(w)\, \Delta w_p^{\mu\nu}}_{\text{Term I: 曲率驱动}} \; - \; \underbrace{(\nabla \Delta w_p^{\mu\nu})^{\top} \nabla \ell_p(w)}_{\text{Term II: } \propto \|\nabla \ell\|}.
\end{aligned}
$$

接近极小值时 $\|\nabla \ell\| \to 0$ 而曲率 $h_p$ 仍显著，Term II 衰减更快，故

$$
g_{\mu\nu} \approx \frac{1}{B} \sum_{p \in \mathcal{B}_\nu} h_p(w)\, \Delta w_p^{\mu\nu}.
$$

（表 1 中 $\gamma_{\text{AWD}}$ 与 $\gamma_{\text{emp}}$ 的微小偏差正来自 MSE 未完全收敛时 Term II 不可忽略。）

### 推导 4：协方差 $=$ 曲率二阶矩（Theorem 3.4）

代入 $C = \frac{1}{2} \mathbb{E}_{\mu\nu}[g_{\mu\nu} g_{\mu\nu}^{\top}]$，i.i.d. 采样使跨样本交叉项消失：

$$
C \approx \frac{1}{2B}\, \mathbb{E}_p\big[ h_p M_p h_p \big],
\qquad M_p = \mathbb{E}_\mu\big[ \Delta w_p^{\mu} (\Delta w_p^{\mu})^{\top} \big].
$$

假设扰动在 $h_p$ 的局部特征基下各向同性且样本间同分布（$M_{p,mn} \approx \sigma_w^2 \delta_{mn}$），在 $H$ 的特征基下取矩阵元：

$$
C_{ij} = v_i^{\top} C v_j \approx \frac{\sigma_w^2}{2B}\, \mathbb{E}_p\!\left[ \sum_m \big(\kappa_m^{(p)}\big)^2 \big(u_m^{(p)} \!\cdot v_i\big) \big(u_m^{(p)} \!\cdot v_j\big) \right],
$$

即 $C \approx \frac{\sigma_w^2}{2B} \mathbb{E}_p[h_p^2]$。与 $H = \mathbb{E}_p[h_p]$ 对比：**一阶矩 vs 二阶矩**，这就是超线性的全部来源。

### 推导 5：普适界 $1 \le \gamma \le 2$（Theorem 5.1）

设局部极小邻域内各 $h_p \succeq 0$ 且特征值 $\le \kappa_{\max}$，记 $\tilde{C}_{ii} := \frac{2B}{\sigma_w^2} C_{ii} = \mathbb{E}_p \|h_p v_i\|^2$。

**上界**（PSD 矩阵的算子不等式 $h_p^2 \preceq \kappa_{\max} h_p$）：

$$
\tilde{C}_{ii} = v_i^{\top}\, \mathbb{E}_p[h_p^2]\, v_i \le \kappa_{\max}\, v_i^{\top}\, \mathbb{E}_p[h_p]\, v_i = \kappa_{\max} H_{ii}.
$$

**下界**（Jensen + Cauchy–Schwarz，$\|v_i\| = 1$）：

$$
\tilde{C}_{ii} = \mathbb{E}_p \|h_p v_i\|^2 \ge \big\| \mathbb{E}_p[h_p]\, v_i \big\|^2 = \|H v_i\|^2 \ge \big( v_i^{\top} H v_i \big)^2 = H_{ii}^2.
$$

两侧界夹出：若经验上 $C_{ii} \propto H_{ii}^{\gamma}$，则 $\gamma \in [1, 2]$。注意下界用的是全局凸性都不需要的初等不等式——只要局部 PSD，比 Fisher 论证的"真参数"假设弱得多。

## 🧪 数值实验

**实验目的：** 用一个玩具模型复现论文第 5.3 节的核心机制——$C_{ii} \propto \mathbb{E}_p[XY]$ 中 $X = \kappa^2$（曲率强度）与 $Y$（方向对齐）的**相关性**决定 $\gamma$：二者独立 $\Rightarrow \gamma = 1$（MSE 情形），正相关 $\Rightarrow \gamma > 1$（CE 情形）。

**实验设置：** 每个样本的 Hessian 取秩-1 形式 $h_p = \kappa_p\, e_{m_p} e_{m_p}^{\top}$（与论文观测一致：逐样本谱由第一特征值主导），方向 $m_p$ 按幂律分布 $\pi_m \propto m^{-a}$ 采样，制造重尾的全局谱。

- **情形 A（模拟 MSE）**：$\kappa_p$ 与方向 $m_p$ 独立 $\Rightarrow$ $C_{ii} = \pi_i\,\mathbb{E}[\kappa^2] \propto H_{ii}$，预期 $\gamma = 1$；
- **情形 B（模拟 CE）**：$\kappa_p \propto \pi_{m_p}$（"越硬的方向越常被激发"，即 $X$–$Y$ 正相关）$\Rightarrow$ $H_{ii} \propto \pi_i^2$，$C_{ii} \propto \pi_i^3$，预期 $\gamma = 1.5$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(42)

In [ ]:
d, N = 100, 200_000          # 维度、样本数
a = 1.5                       # 方向分布的幂律指数 pi_m ∝ m^{-a}
pi = np.arange(1, d + 1) ** (-a)
pi /= pi.sum()

m = rng.choice(d, size=N, p=pi)          # 每个样本的主方向
noise = rng.lognormal(0, 0.3, size=N)    # 曲率的随机涨落

def diag_CH(kappa, m):
    """h_p = kappa_p e_m e_m^T 时，H_ii = E[kappa 1{m=i}], C_ii = E[kappa^2 1{m=i}]"""
    H = np.bincount(m, weights=kappa, minlength=d) / len(m)
    C = np.bincount(m, weights=kappa**2, minlength=d) / len(m)
    return H, C

# 情形 A：kappa 与方向独立（MSE 型）
H_A, C_A = diag_CH(noise, m)
# 情形 B：kappa ∝ pi_m，曲率强度与方向出现频率纠缠（CE 型）
H_B, C_B = diag_CH(noise * pi[m] * d, m)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (H, C, name, expect) in zip(axes, [
        (H_A, C_A, '情形 A：$X\\perp Y$（MSE 型）', 1.0),
        (H_B, C_B, '情形 B：$X$–$Y$ 正相关（CE 型）', 1.5)]):
    mask = (H > 0) & (C > 0)
    x, y = np.log(H[mask]), np.log(C[mask])
    gamma = np.polyfit(x, y, 1)[0]
    ax.loglog(H[mask], C[mask], '.', alpha=0.6)
    ax.set_title(f'{name}\n拟合 $\\gamma$ = {gamma:.2f}（理论 {expect}）')
    ax.set_xlabel('$H_{ii}$'); ax.set_ylabel('$C_{ii}$'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**实验结论：** 

- 情形 A 拟合斜率 $\gamma \approx 1$，情形 B 拟合斜率 $\gamma \approx 1.5$，与手算一致：仅仅改变 $\kappa$（曲率强度）与 $m$（方向）之间的相关结构，就把标度指数从线性推到超线性——复现了论文对 CE vs MSE 差异的解释（"特征值–特征方向纠缠"），且两种情形都落在理论界 $[1, 2]$ 内。
- 待做：换成真实小网络（MNIST + 小 MLP），直接测 $\nabla \ell_i \nabla \ell_i^{\top}$ 的对角元 vs Hessian 对角元（Hutchinson / 有限差分），对比 CE 与 MSE 的 $\gamma$。

## 🤔 疑问与思考

- [ ] $M_{p,mn} \approx \sigma_w^2 \delta_{mn}$（扰动在局部特征基下各向同性、跨样本同方差）是整个 Theorem 3.4 的关键假设，论文中它的经验验证有多强？如果 $\sigma_w^2$ 依赖样本，$C$ 会变成加权二阶矩，界还成立吗（上界成立，下界需要重新看）？
- [ ] CE 中 $X$–$Y$ 正相关的**机理**论文明确留作 open problem（猜测与 KL 散度 vs 二次损失的收敛性质有关，引 Soudry et al. 2018 的 margin 理论）——能否在两层线性网络 + softmax 上解析算出这个相关性？
- [ ] 与 7.18 同读的 WSD/river-valley 论文的联系：河谷图像中"山坡"方向就是大 $H_{ii}$ 方向，$\gamma > 1$ 意味着 CE 训练时山坡方向的噪声被**超线性放大**——这会不会正是 stable 阶段沿河推进的动力来源？值得写个小实验验证。
- [ ] $\gamma$ 是用 top-1000 特征值拟合的；Hessian 谱的退化尾部对幂律的影响如何（论文说为数值稳定性而截断）？

## 🔗 相关工作与延伸阅读

1. Feng, Zhang & Tu, *Activity–weight duality in feed-forward neural networks reveals two co-determinants for generalization*, Nature Machine Intelligence, 2023. —— AWD 框架的出处，必读。
2. Jastrzebski et al., *Three factors influencing minima in SGD*, ICANN 2018；Zhu et al. 2019；Xie et al. 2021. —— 被本文反驳的 $C \propto H$ 经典论证。
3. Feng & Tu, *The inverse variance–flatness relation in SGD*, PNAS 2021. —— 同组前作，噪声方差-平坦度反比关系。
4. Xie et al. 2023; Tang et al. 2025. —— Hessian/协方差谱的重尾幂律结构。
5. Soudry et al. 2018, *The implicit bias of gradient descent on separable data*. —— CE 超线性机理猜想的理论线索。

---

### 📎 当天同读（未做笔记）

- Wen et al., *Understanding Warmup-Stable-Decay Learning Rates: A River Valley Loss Landscape Perspective*, 2024.
- Dauphin et al., *Identifying and Attacking the Saddle Point Problem in High-Dimensional Non-Convex Optimization*, NeurIPS 2014.